# 🤖 AI Engineering Fundamentals — Lezione 3
## Notebook Gruppo B

**ITS Novitas 4.0 | Martedì 26/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "B"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# Setup — eseguite questa cella per prima
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente
print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo B: Conversation History & Multi-turno

Esplorate come funziona la memory del chatbot attraverso la history,
e confrontate le tre strategie di gestione.

---
### Esercizio 1 — Con vs senza history *(guidato)*

La differenza più importante della lezione.
Stessa conversazione, due approcci: solo l'ultimo messaggio vs tutta la history.

In [ ]:
# Esercizio 1 — il chatbot che dimentica vs quello che ricorda

domande = [
    "Mi chiamo Luca e studio AI Engineering a Sassari.",
    "Qual è la capitale della Sardegna?",
    "Come mi chiamo e cosa studio?",  # ← il test della memoria
]

# ── CHATBOT SENZA HISTORY ──────────────────────────────────────────
print("=" * 55)
print("CHATBOT SENZA HISTORY (manda solo l'ultimo messaggio)")
print("=" * 55)

for domanda in domande:
    # Mandiamo SOLO il messaggio corrente: il modello non ha memoria dei turni precedenti
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{"role": "user", "content": domanda}]  # solo l'ultimo
    )
    print(f"👤 {domanda}")
    print(f"🤖 {risposta.content[0].text}\n")

print()

# ── CHATBOT CON HISTORY ────────────────────────────────────────────
print("=" * 55)
print("CHATBOT CON HISTORY (manda tutta la lista)")
print("=" * 55)

history = []
for domanda in domande:
    history.append({"role": "user", "content": domanda})

    # Mandiamo TUTTA la history: il modello "ricorda" i turni precedenti
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=history  # tutta la lista
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    print(f"👤 {domanda}")
    print(f"🤖 {testo}\n")

# Osservazione: alla terza domanda il chatbot SENZA history non sa come ti chiami
# (ogni messaggio è isolato), mentre quello CON history risponde correttamente
# "Luca, studi AI Engineering": la memoria del chatbot è semplicemente la lista
# di messaggi che gli reinviamo ad ogni turno.

---
### Esercizio 2 — Truncation vs Sliding Window *(guidato)*

Fate una conversazione lunga (8 turni) con le due strategie.
Verificate cosa ricorda e cosa dimentica il chatbot in ogni caso.

In [ ]:
# Esercizio 2 — truncation vs sliding window

MAX = 3  # manteniamo solo 3 turni per rendere visibile l'effetto

def chat_truncation(messaggio, history):
    """Tronca la history modificando direttamente la lista."""
    history.append({"role": "user", "content": messaggio})

    # Se la history supera MAX*2 messaggi, eliminiamo i più vecchi tenendo solo gli ultimi MAX*2
    if len(history) > MAX * 2:
        del history[:-MAX * 2]

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

def chat_sliding(messaggio, history):
    """Usa sliding window: la history completa rimane, manda solo una finestra."""
    history.append({"role": "user", "content": messaggio})

    # La history completa resta intatta; al modello inviamo solo gli ultimi MAX*2 messaggi
    messaggi_da_inviare = history[-MAX * 2:]

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=messaggi_da_inviare
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test con 6 turni
turni = [
    "Info 1: Mi chiamo Marco.",
    "Info 2: Lavoro a WiData.",
    "Info 3: Il mio sensore preferito è XS200.",
    "Domanda generica: cos'è il RAG?",
    "Altra domanda generica: cos'è lo streaming?",
    "Test memoria: come mi chiamo, dove lavoro e qual è il mio sensore preferito?",
]

hist_trunc = []
hist_slid  = []

print(f"{'Turno':<8} {'Truncation (history)':<25} {'Sliding (history)':<25}")
print("-" * 60)

for i, turno in enumerate(turni):
    r_t = chat_truncation(turno, hist_trunc)
    r_s = chat_sliding(turno, hist_slid)
    print(f"T{i+1:<7} len={len(hist_trunc):<20} len={len(hist_slid)}")

print("\n--- Risposte al test memoria ---")
print(f"Truncation: {r_t[:150]}")
print(f"Sliding:    {r_s[:150]}")

# Osservazione: con MAX=3, all'ultimo turno le info iniziali (nome, azienda, sensore)
# sono ormai uscite dalla finestra in ENTRAMBE le strategie, quindi il chatbot non
# ricorda più tutti i dettagli. Differenza chiave: con truncation la lista `history`
# viene tagliata davvero (i vecchi messaggi sono persi), mentre con sliding window la
# `history` completa resta salvata e tagliamo solo ciò che inviamo (così possiamo
# recuperare i vecchi turni, ad es. per fare un riassunto — vedi Esercizio 3).

---
### Esercizio 3 — Summarization *(libero)*

Implementate la strategia più avanzata: quando la history
supera una soglia, chiedete a Claude di riassumerla
e usate il riassunto come contesto compresso.

In [ ]:
# Esercizio 3 — summarization

SOGLIA_TURNI = 4  # quando superare questa soglia, riassumi

def riassumi_history(history):
    """Chiede a Claude di riassumere la conversazione in max 100 token."""
    testo_history = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in history
    )
    riassunto = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{
            "role": "user",
            "content": (
                "Riassumi in modo molto conciso la seguente conversazione, "
                "mantenendo nomi, fatti e preferenze importanti dell'utente:\n\n"
                + testo_history
            ),
        }],
    ).content[0].text
    return riassunto

def chat_con_summarization(messaggio, history, riassunto=None):
    """Chatbot con summarization quando la history è troppo lunga."""
    history.append({"role": "user", "content": messaggio})

    # Costruisci il contesto: riassunto (se presente) + ultimi messaggi
    if riassunto and len(history) > SOGLIA_TURNI * 2:
        system = f"Contesto della conversazione precedente: {riassunto}"
        messaggi_da_inviare = history[-4:]  # solo gli ultimi 2 turni
    else:
        system = None
        messaggi_da_inviare = history

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 200,
        "messages": messaggi_da_inviare,
    }
    if system:
        params["system"] = system

    testo = client.messages.create(**params).content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test: conversazione lunga → riassunto → si continua, verificando la memoria
history = []
riassunto = None

turni = [
    "Mi chiamo Anna e sono responsabile IT del Comune di Alghero.",
    "Vorremmo monitorare la qualità dell'aria in centro storico.",
    "Cos'è un sensore PM2.5?",
    "Come funziona la connettività LoRaWAN?",
    "Quanti sensori servono per coprire una piazza?",
    "Riepilogo: come mi chiamo e di quale Comune mi occupo?",  # test memoria DOPO il riassunto
]

for i, turno in enumerate(turni):
    # Quando si supera la soglia, riassumiamo la conversazione fin qui
    if riassunto is None and len(history) >= SOGLIA_TURNI * 2:
        riassunto = riassumi_history(history)
        print(f"📝 Riassunto generato: {riassunto}\n")

    risposta = chat_con_summarization(turno, history, riassunto)
    print(f"👤 {turno}")
    print(f"🤖 {risposta}\n")

# Grazie al riassunto passato come system, il chatbot ricorda ancora che l'utente è Anna
# del Comune di Alghero anche se i primi messaggi non sono più nella finestra inviata.

---
### Esercizio 4 — La strategia giusta per WiData *(libero)*

Un cliente di WiData fa mediamente 15 domande per sessione.
Ogni risposta è lunga circa 100 parole.

Quale strategia di gestione della history consigliate?
Implementatela e motivate la scelta con numeri concreti.

In [ ]:
# Esercizio 4 — la strategia giusta per WiData

# Simulate una sessione tipica WiData: 15 domande sui prodotti
domande_widata = [
    "Quali sensori offrite per ambienti esterni?",
    "Il sensore XS200 funziona con LoRaWAN?",
    "Qual è la durata della batteria?",
    "Come si installa il gateway GW500?",
    "La piattaforma Xplore ha le API REST?",
    "Che certificazioni hanno i vostri sensori?",
    "Posso esportare i dati in CSV?",
    "Ogni quanto vengono aggiornate le misure?",
    "Supportate allarmi via email?",
    "Quanti sensori posso collegare a un gateway?",
    "I dati sono conservati a norma GDPR?",
    "Esiste un'app mobile per Xplore?",
    "Che assistenza offrite dopo l'installazione?",
    "Posso integrare i dati nel mio gestionale?",
    "Riepilogo: di quali argomenti abbiamo parlato finora?",
]

SYSTEM_WIDATA = (
    "Sei l'assistente di WiData, startup IoT di Sassari (monitoraggio ambientale). "
    "Rispondi in italiano, in modo conciso e professionale."
)
MAX_TURNI = 6  # sliding window: ultimi 6 turni (12 messaggi)

def conta_token(messaggi, system=None):
    kwargs = {"model": "claude-haiku-4-5-20251001", "messages": messaggi}
    if system:
        kwargs["system"] = system
    return client.messages.count_tokens(**kwargs).input_tokens

# Strategia scelta: SLIDING WINDOW (+ system prompt fisso, cacheabile)
history = []
token_inviati_totali = 0

for domanda in domande_widata:
    history.append({"role": "user", "content": domanda})
    finestra = history[-MAX_TURNI * 2:]
    token_inviati_totali += conta_token(finestra, system=SYSTEM_WIDATA)

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        system=SYSTEM_WIDATA,
        messages=finestra,
    ).content[0].text
    history.append({"role": "assistant", "content": risposta})

# Confronto: quanti token avremmo inviato SENZA gestione (tutta la history ogni volta)
history_full = []
token_full_totali = 0
for i, domanda in enumerate(domande_widata):
    history_full.append({"role": "user", "content": domanda})
    token_full_totali += conta_token(history_full, system=SYSTEM_WIDATA)
    # riusiamo le risposte già ottenute (indice dispari nella history sliding)
    history_full.append({"role": "assistant", "content": history[i * 2 + 1]["content"]})

costo_sliding = token_inviati_totali / 1_000_000 * 1.0
costo_full = token_full_totali / 1_000_000 * 1.0

print(f"Token inviati con SLIDING WINDOW: {token_inviati_totali}  (${costo_sliding:.6f})")
print(f"Token inviati SENZA gestione:     {token_full_totali}  (${costo_full:.6f})")
print(f"Token risparmiati: {token_full_totali - token_inviati_totali}")
print(f"Costo per 1000 sessioni/mese (sliding): ${costo_sliding * 1000:.4f}")
print(f"Costo per 1000 sessioni/mese (senza):   ${costo_full * 1000:.4f}")

# Confronto finale:
# Strategia scelta: sliding window (ultimi 6 turni) + system prompt fisso.
# Motivazione: con 15 domande la conversazione resta gestibile, ma inviare tutta la
#   history ad ogni turno fa crescere i token in modo quadratico. La sliding window
#   tiene il costo per turno quasi costante mantenendo il contesto recente; il system
#   fisso può essere servito dalla KV Cache. Per non perdere le info iniziali (nome,
#   Comune, ecc.) si combina con un riassunto periodico (Esercizio 3).

---
## 📊 Preparate la presentazione (5 slide)

1. **Con vs senza history** — la differenza mostrata con i vostri risultati
2. **Il pattern corretto** — i 3 passi: aggiungi user, manda tutto, aggiungi assistant
3. **Truncation vs Sliding Window** — quando usa cosa, con i vostri dati
4. **Summarization** — come funziona e quando conviene
5. **La vostra raccomandazione per WiData** — con motivazione numerica

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*